# Off-axis magnetic field $B_z(r, z)$ of a cylindrical permanent magnet

This notebook computes and plots the axial component of the magnetic field $B_z(r, z)$  
as a function of radial distance $r$ for several fixed axial positions $z$.

**Model:** The cylindrical magnet (radius $b$, half-height $l$, remanence $B_r = \mu_0 M$)  
is represented as a surface solenoid — a sheet of current $K = M$ on the cylindrical surface.  
The field is computed by integrating the Biot-Savart law over current rings,  
using exact elliptic integral expressions.

$$B_z(r,z) = \frac{\mu_0 M}{4\pi} \int_{-l}^{+l} \frac{dz'}{\sqrt{(b+r)^2+(z-z')^2}}
\left[ K(k^2) + \frac{b^2-r^2-(z-z')^2}{(b-r)^2+(z-z')^2}\,E(k^2) \right]$$

where $k^2 = \dfrac{4br}{(b+r)^2+(z-z')^2}$ and $K$, $E$ are complete elliptic integrals.

**Key finding:** Near the magnet ($z \lesssim l$), $B_z$ changes sign as a function of $r$ —  
the return flux outside the magnet radius creates a negative $B_z$ annular region.  
This causes the area-averaged $\langle B_z \rangle$ over a pickup coil to be much smaller than the on-axis value.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import ellipk, ellipe
from scipy.integrate import cumulative_trapezoid

## Parameters
Adjust these to match your magnet and pickup coil.

In [ ]:
mu0 = 4 * np.pi * 1e-7   # H/m

# --- Magnet ---
b  = 1e-2    # radius [m]
l  = 1e-2    # half-height [m]
Br = 1.093   # remanence [T]  (fitted from teslameter data)
M  = Br / mu0  # magnetization [A/m]

# --- Pickup coil ---
R_coil = 17.5e-3   # coil radius [m]

print(f"Magnet:  radius b = {b*100:.1f} cm,  half-height l = {l*100:.1f} cm")
print(f"         Br = {Br*1000:.0f} mT,  M = {M:.3e} A/m")
print(f"Pickup coil radius R = {R_coil*100:.1f} cm")

## Core functions

### Single current loop
$B_z$ at point $(r, z)$ from a single ring of radius $a$ at $z'=0$, current $I$:

$$B_z^\text{ring}(r,z;\,a) = \frac{\mu_0 I}{4\pi\sqrt{(a+r)^2+z^2}}
\left[K(k^2) + \frac{a^2-r^2-z^2}{(a-r)^2+z^2}\,E(k^2)\right], \quad
k^2 = \frac{4ar}{(a+r)^2+z^2}$$

In [ ]:
def Bz_single_loop(a, z_rel, r, I=1.0):
    """
    Exact Bz at (r, z_rel) from a current loop of radius a at z'=0.
    Uses complete elliptic integrals K and E.

    Parameters
    ----------
    a     : loop radius [m]
    z_rel : axial distance from loop plane [m]  (z_obs - z_loop)
    r     : radial distance from axis [m]
    I     : current [A]

    Returns
    -------
    Bz [T]
    """
    if r < 1e-12:  # on-axis: analytic limit
        return mu0 * I * a**2 / (2 * (a**2 + z_rel**2)**1.5)

    k2 = 4 * a * r / ((a + r)**2 + z_rel**2)
    k2 = min(k2, 1.0 - 1e-10)   # avoid singularity at k=1 (r=a, z=0)

    K = ellipk(k2)
    E = ellipe(k2)

    prefactor = mu0 * I / (4 * np.pi * np.sqrt((a + r)**2 + z_rel**2))
    bracket   = K + (a**2 - r**2 - z_rel**2) / ((a - r)**2 + z_rel**2) * E

    return prefactor * bracket


def Bz_magnet(z_obs, r_obs, n_loops=600):
    """
    Bz at (r_obs, z_obs) from the cylindrical magnet.
    Integrates over surface current loops from z'=-l to +l.

    Parameters
    ----------
    z_obs   : axial position measured from magnet center [m]
    r_obs   : radial position [m]
    n_loops : number of integration steps along z'

    Returns
    -------
    Bz [T]
    """
    zp  = np.linspace(-l, l, n_loops)
    dzp = zp[1] - zp[0]
    # Each strip dz' carries surface current dI = M * dz'
    Bz = sum(
        Bz_single_loop(b, z_obs - zp_i, r_obs, I=M * dzp)
        for zp_i in zp
    )
    return Bz


def Bz_axis_analytic(z):
    """
    Exact on-axis Bz(r=0, z) — closed form (no elliptic integrals needed).
    Valid for |z| > 0 (avoid division by zero at z=0 for the difference,
    but the function itself is finite everywhere).
    """
    s1 = (z + l) / np.sqrt((z + l)**2 + b**2)
    s2 = (z - l) / np.sqrt((z - l)**2 + b**2)
    return mu0 * M / 2 * (s1 - s2)


def flux_avg_Bz(z_obs, R_coil, n_r=80):
    """
    Area-averaged <Bz> = Phi / (pi R^2) over a circular coil of radius R_coil.
    Phi = integral_0^R Bz(r, z) * 2*pi*r dr
    """
    r_pts = np.linspace(0, R_coil, n_r)
    integrand = np.array([Bz_magnet(z_obs, r) * 2 * np.pi * r for r in r_pts])
    Phi = np.trapz(integrand, r_pts)
    return Phi / (np.pi * R_coil**2)


# Quick sanity check
print("On-axis Bz at z=0 (analytic): ", Bz_axis_analytic(0)*1000, "mT")
print("On-axis Bz at z=0 (numeric): ",  Bz_magnet(0, 0)*1000,      "mT")
print("On-axis Bz at z=2cm:         ",  Bz_axis_analytic(2e-2)*1000, "mT")

## Compute $B_z(r)$ at fixed $z$ values

The radial grid is made finer near $r = b$ (magnet edge) where the field changes rapidly.

In [ ]:
# Radial grid: finer near magnet edge r=b
r_arr = np.concatenate([
    np.linspace(0,       0.0088,  40),   # inside magnet radius
    np.linspace(0.0090,  0.0112,  40),   # near the edge  <-- dense
    np.linspace(0.012,   0.05,    80),   # outside
])

z_values_cm = [0, 1, 2, 3, 4, 5]   # z in cm
colors = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#3498db', '#9b59b6']

print("Computing Bz(r) profiles ... (takes ~1 min)")
results = {}
for z_cm in z_values_cm:
    z = z_cm * 1e-2
    Bz = np.array([Bz_magnet(z, r) for r in r_arr])
    results[z_cm] = Bz
    Bz_at_coil = float(np.interp(R_coil, r_arr, Bz))
    print(f"  z={z_cm} cm:  B(r=0)={Bz[0]*1000:7.1f} mT   "
          f"B(r=b)={np.interp(b, r_arr, Bz)*1000:7.1f} mT   "
          f"B(r=R)={Bz_at_coil*1000:7.1f} mT"
          + ("  ← sign change!" if Bz[0]*Bz_at_coil < 0 else ""))
print("Done.")

## Plot 1: Linear scale

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for z_cm, col in zip(z_values_cm, colors):
    ax.plot(r_arr * 100, results[z_cm] * 1000,
            color=col, lw=2, label=f'z = {z_cm} cm')

ax.axvline(b * 100,      color='black', lw=1.5, ls='--', label=f'Magnet edge  b = {b*100:.0f} cm')
ax.axvline(R_coil * 100, color='gray',  lw=1.5, ls=':',  label=f'Coil radius  R = {R_coil*100:.1f} cm')
ax.axhline(0, color='black', lw=0.8, alpha=0.4)

ax.set_xlabel('Radial distance  $r$  [cm]', fontsize=12)
ax.set_ylabel('$B_z(r, z)$  [mT]', fontsize=12)
ax.set_title('Axial field component $B_z$ as a function of $r$ — linear scale', fontsize=12)
ax.legend(fontsize=9, loc='upper right')
ax.grid(True, alpha=0.4)
ax.set_xlim(0, 5)
plt.tight_layout()
plt.show()

## Plot 2: Log scale

Solid lines: $B_z > 0$.  Dashed lines: $|B_z|$ where $B_z < 0$ (return flux region).

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for z_cm, col in zip(z_values_cm, colors):
    Bz = results[z_cm]
    pos = np.where(Bz > 0, Bz, np.nan)
    neg = np.where(Bz < 0, -Bz, np.nan)
    ax.semilogy(r_arr*100, pos*1000, color=col, lw=2,   label=f'z = {z_cm} cm')
    ax.semilogy(r_arr*100, neg*1000, color=col, lw=1.5, ls='--', alpha=0.7)

ax.axvline(b * 100,      color='black', lw=1.5, ls='--', label=f'Magnet edge  b = {b*100:.0f} cm')
ax.axvline(R_coil * 100, color='gray',  lw=1.5, ls=':',  label=f'Coil radius  R = {R_coil*100:.1f} cm')

ax.set_xlabel('Radial distance  $r$  [cm]', fontsize=12)
ax.set_ylabel('$|B_z(r, z)|$  [mT]', fontsize=12)
ax.set_title('Axial field $B_z$ — log scale\n'
             '(solid: $B_z > 0$,  dashed: $|B_z|$ where $B_z < 0$)', fontsize=12)
ax.legend(fontsize=9, loc='upper right')
ax.grid(True, alpha=0.4, which='both')
ax.set_xlim(0, 5)
ax.set_ylim(0.1, 2000)
plt.tight_layout()
plt.show()

## Plot 3: Area-averaged $\langle B_z\rangle$ vs on-axis $B_z$ as a function of $z$

The pickup coil measures the flux average $\langle B_z\rangle_R = \Phi/(\pi R^2)$,  
while the teslameter measures $B_z(r=0, z)$. This plot shows why they differ.

In [ ]:
from scipy.special import ellipk, ellipe

def flux_through_disc(z_magnet, R_disc, n_src=400):
    """
    Total flux Phi through a disc of radius R_disc at height z_magnet
    above the magnet center. Uses the mutual inductance formula.
    """
    zp  = np.linspace(-l, l, n_src)
    dzp = zp[1] - zp[0]
    Phi = 0.0
    for zp_i in zp:
        z_rel = z_magnet - zp_i
        a = b
        k2 = 4*a*R_disc / ((a + R_disc)**2 + z_rel**2)
        k2 = min(k2, 1 - 1e-10)
        k  = np.sqrt(k2)
        K  = ellipk(k2);  E = ellipe(k2)
        M_mutual = mu0 * np.sqrt(a * R_disc) * ((2/k - k)*K - (2/k)*E)
        Phi += M_mutual * (M * dzp)
    return Phi

z_arr  = np.linspace(0, 0.12, 40)
B_axis = np.array([Bz_axis_analytic(z) for z in z_arr])
B_avg  = np.array([flux_through_disc(z, R_coil)/(np.pi*R_coil**2) for z in z_arr])
ratio  = B_avg / B_axis

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: B_axis and <Bz> on log scale
ax = axes[0]
ax.semilogy(z_arr*100, B_axis*1000, 'r-',  lw=2, label='$B_z$ on axis (teslameter)')
ax.semilogy(z_arr*100, B_avg*1000,  'b-',  lw=2, label=r'$\langle B_z\rangle$ over coil area')
ax.axvline(l*100,      color='gray', ls='--', lw=1, label=f'Magnet face z=l={l*100:.0f} cm')
ax.axvline(R_coil*100, color='orange', ls=':', lw=1, label=f'z = R_coil = {R_coil*100:.1f} cm')
ax.set_xlabel('z  [cm]', fontsize=12)
ax.set_ylabel('B  [mT]', fontsize=12)
ax.set_title('On-axis vs flux-averaged $B_z$', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.4, which='both')
ax.set_xlim(0, 12)

# Right: ratio
ax = axes[1]
ax.plot(z_arr*100, ratio, 'b-', lw=2)
ax.axhline(1.0, color='gray', ls='--', lw=1, label='ratio = 1 (perfect agreement)')
ax.axhline(0.5, color='orange', ls=':', lw=1, label='ratio = 0.5 (dipole limit)')
ax.axvline(l*100,      color='gray', ls='--', lw=1)
ax.axvline(R_coil*100, color='orange', ls=':', lw=1)
# Mark z values used in Bz(r) plots
for z_cm, col in zip(z_values_cm, colors):
    r_val = np.interp(z_cm*1e-2, z_arr, ratio)
    ax.plot(z_cm, r_val, 'o', color=col, ms=8)
    ax.annotate(f'z={z_cm}cm', xy=(z_cm, r_val),
                xytext=(z_cm+0.2, r_val+0.03), fontsize=8, color=col)
ax.set_xlabel('z  [cm]', fontsize=12)
ax.set_ylabel(r'$\langle B_z\rangle\,/\,B_\mathrm{axis}$', fontsize=12)
ax.set_title(r'Ratio $\langle B_z\rangle / B_\mathrm{axis}$', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.4)
ax.set_xlim(0, 12)
ax.set_ylim(0, 1.1)

plt.tight_layout()
plt.show()

## Summary table

Values of $B_z$ at the axis, magnet edge, and coil radius for each $z$.

In [ ]:
print(f"{'z [cm]':>8}  {'B(r=0) [mT]':>13}  {'B(r=b) [mT]':>13}  "
      f"{'B(r=R) [mT]':>13}  {'sign change?':>14}")
print("-" * 70)

for z_cm in z_values_cm:
    Bz  = results[z_cm]
    b0  = Bz[0]
    bb  = float(np.interp(b,      r_arr, Bz))
    bR  = float(np.interp(R_coil, r_arr, Bz))
    chg = "YES ← return flux" if b0 * bR < 0 else ""
    print(f"{z_cm:>8}  {b0*1000:>13.1f}  {bb*1000:>13.1f}  {bR*1000:>13.1f}  {chg:>14}")

## Physical interpretation

**Why does $B_z$ change sign at small $z$?**

Magnetic field lines form closed loops ($\nabla\cdot\mathbf{B}=0$). The strong axial flux going
through the magnet center must return somewhere. At $z \approx 0$ and $z \approx l$, the return
flux passes radially outward just beyond the magnet radius $r > b$, creating a region where
$B_z < 0$.

**Consequence for the pickup coil:**

When the coil radius $R > b$, the coil integrates over both the positive on-axis region
and the negative return-flux annulus. The contributions partially cancel, making
$\langle B_z\rangle_R \ll B_z(0)$. This is the root cause of the factor-of-3–4 discrepancy
between dynamic (coil) and static (teslameter) magnetic field measurements near $z = 0$.